# 05 Richardson 外推与隐式差分格式

第四章中 Romberg 求积用 Richardson 外推提高梯形公式的精度。本节把同一思想用于数值微分，并建立隐式紧致差分格式的入口。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import central_difference, compact_first_derivative_periodic, richardson_derivative, richardson_extrapolate


## Richardson 外推公式

若差分近似满足

$$
D(h)=f'(x)+C h^p+O(h^{p+q}),
$$

则

$$
D(h/2)=f'(x)+C\frac{h^p}{2^p}+O(h^{p+q}).
$$

消去主误差项得到

$$
D_{\mathrm{ext}}=\frac{2^pD(h/2)-D(h)}{2^p-1}.
$$

中心差分的主误差阶通常是 $p=2$，并且误差展开只含偶次幂。


In [ ]:
def teaching_richardson_table(values, base_order=2, order_step=2):
    values = np.asarray(values, dtype=float)
    n = values.size
    table = np.full((n, n), np.nan)
    table[:, 0] = values
    for col in range(1, n):
        factor = 2 ** (base_order + (col - 1) * order_step)
        for row in range(col, n):
            table[row, col] = table[row, col - 1] + (table[row, col - 1] - table[row - 1, col - 1]) / (factor - 1)
    return table

x0 = 0.7
h0 = 0.25
steps = h0 / (2.0 ** np.arange(5))
values = np.array([float(central_difference(np.sin, x0, h)) for h in steps])
teaching_table = teaching_richardson_table(values)
print(teaching_table)
print("exact=", math.cos(x0))


## 与正式实现对照

`richardson_derivative` 会生成步长序列、基础差分结果和外推表。它适合教学和实验，不试图自动判断最佳层数；外推层数过多时，舍入误差可能被放大。


In [ ]:
result = richardson_derivative(np.sin, x0, h=h0, levels=5)
print("步长：", result.step_sizes)
print("外推表：")
print(result.table)
print("外推结果：", result.value)
print("误差：", abs(result.value - math.cos(x0)))


## 何时外推不再有效

Richardson 外推依赖规则的误差展开和足够准确的函数值。若函数不够光滑、步长已经进入舍入误差主导区域，或数据本身含噪声，外推可能不再改进结果。


In [ ]:
x0 = 1.0
exact = math.exp(x0)
base_steps = 10.0 ** (-np.arange(1, 9, dtype=float))
errors = []
for h in base_steps:
    value = richardson_derivative(np.exp, x0, h=h, levels=3).value
    errors.append(abs(value - exact))

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(base_steps, errors, marker="o")
ax.invert_xaxis()
ax.set_xlabel("初始步长 h")
ax.set_ylabel("三层外推绝对误差")
ax.set_title("外推也会受到舍入误差限制")
ax.grid(True, which="both", alpha=0.3)
plt.show()


## 显式差分与隐式紧致差分

显式中心差分直接写成

$$
d_i=\frac{f_{i+1}-f_{i-1}}{2h}.
$$

紧致差分让相邻导数值共同满足方程，例如周期网格上

$$
\frac14 d_{i-1}+d_i+\frac14 d_{i+1}
=\frac{3}{4h}(f_{i+1}-f_{i-1}).
$$

这会形成一个线性方程组。它用较短模板获得较高精度，但需要求解系统并处理边界条件。


In [ ]:
def teaching_compact_periodic(y, h):
    y = np.asarray(y, dtype=float)
    n = y.size
    A = np.eye(n)
    for i in range(n):
        A[i, (i - 1) % n] = 0.25
        A[i, (i + 1) % n] = 0.25
    rhs = 0.75 * (np.roll(y, -1) - np.roll(y, 1)) / h
    return np.linalg.solve(A, rhs)

n = 64
x = np.linspace(0.0, 2.0 * math.pi, n, endpoint=False)
h = x[1] - x[0]
y = np.sin(x)
explicit = (np.roll(y, -1) - np.roll(y, 1)) / (2 * h)
compact = teaching_compact_periodic(y, h)
print("显式中心最大误差：", np.max(np.abs(explicit - np.cos(x))))
print("紧致差分最大误差：", np.max(np.abs(compact - np.cos(x))))
print("正式实现最大误差：", np.max(np.abs(compact_first_derivative_periodic(y, h) - np.cos(x))))


## 边界条件说明

上面的紧致格式使用周期边界，所以第一个节点和最后一个节点相邻。一般区间 $[a,b]$ 上还需要补充端点方程，例如使用单边高阶公式或特定边界条件。本轮只实现周期示例，非周期紧致格式作为后续待完善内容。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, np.cos(x), label="解析导数")
ax.plot(x, explicit, "--", label="显式中心差分")
ax.plot(x, compact, ":", linewidth=2, label="周期紧致差分")
ax.set_title("显式差分与隐式紧致差分比较")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 小结

Richardson 外推通过多个步长消去主误差项，和第四章 Romberg 求积完全同源。紧致差分则通过线性系统把相邻导数值耦合起来，以较小模板获得较高精度。两者都能提高精度，但都不能绕开光滑性、舍入误差和边界条件这些基本限制。
